# Row-Level Security Setup — Community Health Intelligence Platform

This notebook implements **county-level and sub-county-level Row-Level Security (RLS)** using Unity Catalog row filters on gold layer tables in `community_health_intelligence.gold`.

### How It Works
- A `user_access_control` mapping table associates each Databricks user with a geographic scope
- SQL UDFs evaluate `CURRENT_USER()` against this mapping at query time
- Row filters applied to each gold table restrict rows to the user's assigned geography

### Access Levels
| Level | Sees | Example |
|-------|------|--------|
| **ADMIN** | All data across all counties/sub-counties | keyegonaws@gmail.com |
| **COUNTY** | Only rows in their assigned county | keyegon@gmail.com (BUSIA) |
| **SUBCOUNTY** | Only rows in their assigned sub-county | erickkiprotichyegon61@gmail.com (TESO NORTH) |

### Tables Protected (11)
- **Dimensions:** dim_chw, dim_facility, dim_geography
- **Facts:** fact_family_planning, fact_home_visit, fact_immunization, fact_pnc, fact_population, fact_pregnancy, fact_pregnancy_journey, fact_supervision

### Views (Auto-Inherit)
- mv_family_planning, mv_immunization, mv_maternal_health, mv_supervision
- vw_chw_performance, vw_coverage_gaps, vw_executive_summary, vw_maternal_funnel

In [0]:
%sql
-- =============================================================
-- STEP 1: Create the user-access mapping table
-- =============================================================
CREATE TABLE IF NOT EXISTS community_health_intelligence.gold.user_access_control (
  user_email STRING NOT NULL COMMENT 'Databricks user email (matches CURRENT_USER())',
  access_level STRING NOT NULL COMMENT 'ADMIN, COUNTY, or SUBCOUNTY',
  county_name STRING COMMENT 'Assigned county (NULL for ADMIN)',
  sub_county_name STRING COMMENT 'Assigned sub-county (NULL for ADMIN/COUNTY level)',
  created_at TIMESTAMP,
  updated_at TIMESTAMP
)
COMMENT 'Maps users to their geographic data access scope for row-level security'
TBLPROPERTIES ('quality' = 'gold', 'purpose' = 'row-level-security');

In [0]:
%sql
-- =============================================================
-- STEP 2: Seed user access mappings
-- ⚠️ Replace example emails with real users before production
-- =============================================================
INSERT INTO community_health_intelligence.gold.user_access_control
  (user_email, access_level, county_name, sub_county_name, created_at, updated_at)
VALUES
  -- Real users
  (CURRENT_USER(), 'ADMIN', NULL, NULL, CURRENT_TIMESTAMP(), CURRENT_TIMESTAMP()),
  -- Example users
  ('busia.manager@example.com', 'COUNTY', 'BUSIA', NULL, CURRENT_TIMESTAMP(), CURRENT_TIMESTAMP()),
  ('kisumu.manager@example.com', 'COUNTY', 'KISUMU', NULL, CURRENT_TIMESTAMP(), CURRENT_TIMESTAMP()),
  ('teso.north.officer@example.com', 'SUBCOUNTY', 'BUSIA', 'TESO NORTH', CURRENT_TIMESTAMP(), CURRENT_TIMESTAMP()),
  ('matayos.officer@example.com', 'SUBCOUNTY', 'BUSIA', 'MATAYOS', CURRENT_TIMESTAMP(), CURRENT_TIMESTAMP()),
  ('kisumu.central.officer@example.com', 'SUBCOUNTY', 'KISUMU', 'KISUMU CENTRAL', CURRENT_TIMESTAMP(), CURRENT_TIMESTAMP()),
  ('admin@example.com', 'ADMIN', NULL, NULL, CURRENT_TIMESTAMP(), CURRENT_TIMESTAMP());

SELECT * FROM community_health_intelligence.gold.user_access_control ORDER BY access_level, county_name;

In [0]:
%sql
-- =============================================================
-- STEP 2b: Add real production users
-- Uses MERGE to avoid duplicates if users already exist
-- =============================================================

MERGE INTO community_health_intelligence.gold.user_access_control AS target
USING (
  SELECT * FROM VALUES
    ('keyegonaws@gmail.com',              'ADMIN',     NULL,    NULL,         CURRENT_TIMESTAMP(), CURRENT_TIMESTAMP()),
    ('keyegon@gmail.com',                 'COUNTY',    'BUSIA', NULL,         CURRENT_TIMESTAMP(), CURRENT_TIMESTAMP()),
    ('erickkiprotichyegon61@gmail.com',   'SUBCOUNTY', 'BUSIA', 'TESO NORTH', CURRENT_TIMESTAMP(), CURRENT_TIMESTAMP())
  AS src(user_email, access_level, county_name, sub_county_name, created_at, updated_at)
) AS source
ON target.user_email = source.user_email
WHEN NOT MATCHED THEN
  INSERT (user_email, access_level, county_name, sub_county_name, created_at, updated_at)
  VALUES (source.user_email, source.access_level, source.county_name, source.sub_county_name, source.created_at, source.updated_at);

-- Verify all users
SELECT user_email, access_level, county_name, sub_county_name
FROM community_health_intelligence.gold.user_access_control
ORDER BY access_level, county_name, sub_county_name;

user_email,access_level,county_name,sub_county_name
keyegonaws@gmail.com,ADMIN,null,null
admin@example.com,ADMIN,null,null
keyegon@gmail.com,COUNTY,BUSIA,null
busia.manager@example.com,COUNTY,BUSIA,null
kisumu.manager@example.com,COUNTY,KISUMU,null
matayos.officer@example.com,SUBCOUNTY,BUSIA,MATAYOS
teso.north.officer@example.com,SUBCOUNTY,BUSIA,TESO NORTH
erickkiprotichyegon61@gmail.com,SUBCOUNTY,BUSIA,TESO NORTH
kisumu.central.officer@example.com,SUBCOUNTY,KISUMU,KISUMU CENTRAL


In [0]:
%sql
-- =============================================================
-- STEP 3: County-level row filter function
-- ADMIN sees all; COUNTY/SUBCOUNTY see their assigned county
-- =============================================================
CREATE OR REPLACE FUNCTION community_health_intelligence.gold.county_access_filter(county STRING)
RETURN
  IF(
    EXISTS(
      SELECT 1 FROM community_health_intelligence.gold.user_access_control
      WHERE user_email = CURRENT_USER() AND access_level = 'ADMIN'
    ),
    TRUE,
    EXISTS(
      SELECT 1 FROM community_health_intelligence.gold.user_access_control
      WHERE user_email = CURRENT_USER() AND county_name = county
    )
  );

In [0]:
%sql
-- =============================================================
-- STEP 4: Sub-county cascading row filter function
-- ADMIN → all | COUNTY → their county | SUBCOUNTY → their sub-county
-- =============================================================
CREATE OR REPLACE FUNCTION community_health_intelligence.gold.subcounty_access_filter(county STRING, sub_county STRING)
RETURN
  IF(
    EXISTS(
      SELECT 1 FROM community_health_intelligence.gold.user_access_control
      WHERE user_email = CURRENT_USER() AND access_level = 'ADMIN'
    ),
    TRUE,
    IF(
      EXISTS(
        SELECT 1 FROM community_health_intelligence.gold.user_access_control
        WHERE user_email = CURRENT_USER() AND access_level = 'COUNTY' AND county_name = county
      ),
      TRUE,
      EXISTS(
        SELECT 1 FROM community_health_intelligence.gold.user_access_control
        WHERE user_email = CURRENT_USER() AND access_level = 'SUBCOUNTY'
          AND county_name = county AND sub_county_name = sub_county
      )
    )
  );

In [0]:
%sql
-- =============================================================
-- STEP 5: Apply row filters to ALL gold layer tables
-- =============================================================

-- Dimension tables (county_name, sub_county_name)
ALTER TABLE community_health_intelligence.gold.dim_chw
  SET ROW FILTER community_health_intelligence.gold.subcounty_access_filter ON (county_name, sub_county_name);

ALTER TABLE community_health_intelligence.gold.dim_facility
  SET ROW FILTER community_health_intelligence.gold.subcounty_access_filter ON (county_name, sub_county_name);

ALTER TABLE community_health_intelligence.gold.dim_geography
  SET ROW FILTER community_health_intelligence.gold.subcounty_access_filter ON (county_name, sub_county_name);

-- Fact tables (county_clean, sub_county_clean)
ALTER TABLE community_health_intelligence.gold.fact_family_planning
  SET ROW FILTER community_health_intelligence.gold.subcounty_access_filter ON (county_clean, sub_county_clean);

ALTER TABLE community_health_intelligence.gold.fact_home_visit
  SET ROW FILTER community_health_intelligence.gold.subcounty_access_filter ON (county_clean, sub_county_clean);

ALTER TABLE community_health_intelligence.gold.fact_immunization
  SET ROW FILTER community_health_intelligence.gold.subcounty_access_filter ON (county_clean, sub_county_clean);

ALTER TABLE community_health_intelligence.gold.fact_pnc
  SET ROW FILTER community_health_intelligence.gold.subcounty_access_filter ON (county_clean, sub_county_clean);

ALTER TABLE community_health_intelligence.gold.fact_population
  SET ROW FILTER community_health_intelligence.gold.subcounty_access_filter ON (county_clean, sub_county_clean);

ALTER TABLE community_health_intelligence.gold.fact_pregnancy
  SET ROW FILTER community_health_intelligence.gold.subcounty_access_filter ON (county_clean, sub_county_clean);

ALTER TABLE community_health_intelligence.gold.fact_pregnancy_journey
  SET ROW FILTER community_health_intelligence.gold.subcounty_access_filter ON (county_clean, sub_county_clean);

ALTER TABLE community_health_intelligence.gold.fact_supervision
  SET ROW FILTER community_health_intelligence.gold.subcounty_access_filter ON (county_clean, sub_county_clean);

In [0]:
%sql
-- =============================================================
-- STEP 6: Verify RLS status across all gold objects
-- =============================================================
SELECT
  t.table_name,
  t.table_type,
  CASE
    WHEN t.table_name IN ('dim_chw','dim_facility','dim_geography',
                          'fact_family_planning','fact_home_visit','fact_immunization',
                          'fact_pnc','fact_population','fact_pregnancy',
                          'fact_pregnancy_journey','fact_supervision')
    THEN '✅ subcounty_access_filter'
    WHEN t.table_name LIKE 'mv_%' THEN '🔗 Inherits via source tables'
    WHEN t.table_name LIKE 'vw_%' THEN '🔗 Inherits via source tables'
    WHEN t.table_name = 'user_access_control' THEN '🔒 Access control table'
    ELSE '⬜ No filter (no county column)'
  END AS rls_status
FROM community_health_intelligence.information_schema.tables t
WHERE t.table_schema = 'gold'
ORDER BY rls_status DESC, t.table_name;

In [0]:
%sql
-- =============================================================
-- STEP 7: Smoke test — verify data access
-- ADMIN should see all counties; COUNTY users see only theirs
-- =============================================================
SELECT 'dim_chw' AS tbl, county_name, COUNT(*) AS rows
FROM community_health_intelligence.gold.dim_chw GROUP BY county_name
UNION ALL
SELECT 'fact_pregnancy', county_clean, COUNT(*)
FROM community_health_intelligence.gold.fact_pregnancy GROUP BY county_clean
UNION ALL
SELECT 'fact_immunization', county_clean, COUNT(*)
FROM community_health_intelligence.gold.fact_immunization GROUP BY county_clean
ORDER BY tbl, county_name;

In [0]:
%sql
-- =============================================================
-- ⚠️ EMERGENCY ROLLBACK: Drop ALL row filters
-- Only UNCOMMENT and run to disable RLS entirely
-- =============================================================

-- ALTER TABLE community_health_intelligence.gold.dim_chw DROP ROW FILTER;
-- ALTER TABLE community_health_intelligence.gold.dim_facility DROP ROW FILTER;
-- ALTER TABLE community_health_intelligence.gold.dim_geography DROP ROW FILTER;
-- ALTER TABLE community_health_intelligence.gold.fact_family_planning DROP ROW FILTER;
-- ALTER TABLE community_health_intelligence.gold.fact_home_visit DROP ROW FILTER;
-- ALTER TABLE community_health_intelligence.gold.fact_immunization DROP ROW FILTER;
-- ALTER TABLE community_health_intelligence.gold.fact_pnc DROP ROW FILTER;
-- ALTER TABLE community_health_intelligence.gold.fact_population DROP ROW FILTER;
-- ALTER TABLE community_health_intelligence.gold.fact_pregnancy DROP ROW FILTER;
-- ALTER TABLE community_health_intelligence.gold.fact_pregnancy_journey DROP ROW FILTER;
-- ALTER TABLE community_health_intelligence.gold.fact_supervision DROP ROW FILTER;
-- DROP FUNCTION community_health_intelligence.gold.county_access_filter;
-- DROP FUNCTION community_health_intelligence.gold.subcounty_access_filter;

## Production Deployment Checklist

- [x] Create `user_access_control` mapping table
- [x] Create `county_access_filter` function
- [x] Create `subcounty_access_filter` function (cascading)
- [x] Apply row filters to 11 gold tables (3 dim + 8 fact)
- [x] Verify ADMIN sees all data
- [x] Add real users (keyegonaws, keyegon, erickkiprotichyegon61)
- [ ] Test with COUNTY user (keyegon@gmail.com) — should see only BUSIA
- [ ] Test with SUBCOUNTY user (erickkiprotichyegon61@gmail.com) — should see only TESO NORTH
- [ ] Remove example @example.com users for production
- [ ] Document RLS policies in data governance catalog
- [ ] Monitor query performance after RLS activation

### Managing Users
```sql
-- Add a new user
INSERT INTO community_health_intelligence.gold.user_access_control
VALUES ('new.user@company.com', 'COUNTY', 'BUSIA', NULL, CURRENT_TIMESTAMP(), CURRENT_TIMESTAMP());

-- Remove a user
DELETE FROM community_health_intelligence.gold.user_access_control
WHERE user_email = 'old.user@company.com';

-- Change access level
UPDATE community_health_intelligence.gold.user_access_control
SET access_level = 'ADMIN', county_name = NULL, sub_county_name = NULL, updated_at = CURRENT_TIMESTAMP()
WHERE user_email = 'promoted.user@company.com';
```